# Discover Page 登録内容：同一機種判定ルール

> Discover Pages は Beta 機能です。画面の項目名・配置は変わる可能性があります。
> 機密原価・個人情報・規制対象データは記載しません。

## 登録フィールド

※ テーブル名は Free Edition の既定値（`workspace.vehicle_alias_handson`）です。`config/00_config` で変更した場合は、自分のカタログ・スキーマ名に読み替えてください。

| 項目 | 入力値 |
|---|---|
| Domain | `Vehicle Profitability` |
| Page名 | `同一機種判定ルール` |
| Description | 国・部門・システムごとに異なる車両名称・機種コードを、同じ共通機種IDに対応づけてよいかを判断するための業務ルール |
| Synonyms | `同一機種判定` / `機種対応ルール` / `名寄せルール` / `共通機種ID` / `canonical vehicle` / `vehicle mapping rule` |
| Related assets | `workspace.vehicle_alias_handson.vehicle_alias_master`<br>`workspace.vehicle_alias_handson.alias_mapping_history`<br>`workspace.vehicle_alias_handson.v_alias_approved`<br>`workspace.vehicle_alias_handson.v_alias_review_queue`<br>`workspace.vehicle_alias_handson.v_dq_summary`<br>`workspace.vehicle_alias_handson.alias_evidence`<br>`workspace.vehicle_alias_handson.evidence_weight`<br>`workspace.vehicle_alias_handson.v_evidence_score` |
| Sources | 「車両名称・コード名寄せ運用ルール VPR-RULE-001 Rev.1（架空）」 |

---

## 本文（以下をそのまま Page 本文に貼り付け）

### 目的

販売・生産・開発の各システムは、同じ車両に対して別の名称・コードを使っています。このPageは、それらを **共通機種ID** に対応づけてよいかを判断するためのルールです。

**基本方針：誤って結合するよりも、「要確認」として残すことを優先します。**

### 役割分担

| 仕組み | 役割 | 使いどころ |
|---|---|---|
| Discover Page（このPageと各車両Page） | 業務上の意味・定義・判断根拠を人とGenieに伝える | 「これは同じ車両か」「なぜそう判断したか」への回答 |
| `vehicle_alias_master`（Deltaテーブル） | コードから共通機種IDへの **確定的な変換** | 計画・販売・生産の結合と集計 |
| データ品質ルール（SQL / Lakeflow の期待値） | 不整合を **決定的に検出** する | タイヤ本数・重複・マスタ矛盾・欠損のチェック |
| AIによる候補生成 | 担当者の判断を **支援** する | 未解決コードの候補とスコアの提示（確定はしない） |

Pageを書いても、データの結合やコード変換が自動で実行されるわけではありません。変換は必ず `vehicle_alias_master` を使って行います。

### 判定の手順

1. **完全一致**：元コードが確定コードと一致する。
2. **正規化一致**：全角を半角に、小文字を大文字に変え、空白・記号を取り除いたうえで確定コードと一致する（例：`ＪＰ－Ａ１１`、`jp-a11 ` → `JPA11`）。
3. **過去の承認履歴**：人手で作成した対応履歴のうち、承認済みで、コードと適用期間があるものを対応表に取り込んで使う（例：`CN-X123B`）。
4. **候補**：1〜3で決まらない場合は、AIやルールで候補とスコアを作り、担当者が確認する。
5. **確定**：担当者が承認した対応だけが `approval_status = 'APPROVED'` となり、集計に使われる。

1〜3 はいずれも、次の条件をすべて満たす場合に限ります。
- 業務領域（販売・開発・生産）が一致する
- 対象データの年月が適用期間内にある
- 一致する共通機種IDが1つだけである

### 信頼度による分類

| 信頼度スコア | 分類 | 扱い |
|---|---|---|
| 95点以上 | 自動承認候補 | 担当者が根拠を確認し、ワンクリックで承認できる。**承認操作と承認記録は必須** |
| 70〜94点 | 人による確認 | 担当者が仕様書・過去履歴で確認してから承認または却下する |
| 70点未満 | 未解決 | 集計から除外し、未解決一覧で管理する |
| 候補が複数ある | 人による確認 | スコアに関わらず人が判断する（すべて70点未満なら未解決） |
| 1つのコードが複数の共通機種IDに確定している | 人による確認（マスタ矛盾） | 両方を集計から除外し、マスタを是正する |

### 同じ共通機種IDとしてよいもの

- 同じ世代で、国や部門によって名称・コードが違うもの
- 同じ世代で、システム移行などにより期間ごとにコードが変わったもの（適用期間で切り替える）
- 同じ世代で、パワートレインやドア形状だけが違うもの（属性列で区別する）

### 別の共通機種IDとして扱うもの

- 世代が違うもの（例：Alpha 10th Gen と Alpha 11th Gen）
- 同じ世代でも、別の開発プロジェクト・開発コードを持つ派生車（例：派生EV）
- 市場向けに仕様が大きく異なり、別の開発コードが付いているもの

### 判断してはいけない根拠

- 名称が一致するだけ（同じ名称が複数の世代で使われている）
- コードの見た目が似ているだけ（桁落ち・入力ミスの可能性がある）
- AIの類似度スコアだけ
- 適用期間が切れた対応

### 根拠による候補スコア

未解決のコードについては、属人的に管理されていた情報や利用履歴を**根拠**として集めます。根拠は `alias_evidence` に、スコアは `v_evidence_score` にあります。

| 根拠 | 情報源の系統 | 重み |
|---|---|---:|
| 出荷実績で、確定済みの MTO がそのコードとして出荷されている | 取引 | 40 |
| 過去の原価シートで、確定済みのコードと同じ行グループで合計されていた | 資料 | 25 |
| 担当者メモ・ヒアリングで、そのコードと一意に決まる車両名が一緒に書かれている | 人 | 15 |
| 分析クエリで、確定済みのコードと一緒に使われていた | 利用 | 8 |

- 同じ種類の根拠は、何件あっても1回だけ加点します。
- **系統の違う根拠が2つ以上そろわない限り、69点を上限**とします。
- 利用履歴は、過去の誤った結合も拾うため、最も弱い根拠として扱います。
- 複数の世代で使われている名称（例：Alpha）は、メモの手がかりに使いません。
- スコアが上がっても確定ではありません。上の「信頼度による分類」にしたがって、人が承認します。
- 70点に届かない候補は、足りない根拠の種類を示して、次に集める情報を決めます。

### 承認と記録

- 承認者は **役割名** で記録します（例：「商品企画部 マスタ管理担当」）。
- 承認時には、根拠資料（仕様管理資料の番号など）を `source_document` に記録します。
- `vehicle_alias_master` の変更履歴は Delta の履歴（`DESCRIBE HISTORY`）で確認できます。

### データ品質ルール（違反件数は `v_dq_summary` で確認）

| ID | ルール | 重要度 |
|---|---|---|
| DQ-1 | 1台当たりのタイヤ数量が4本を超えていないこと | HIGH |
| DQ-2 | 同一MTOで同じ部品が重複していないこと | HIGH |
| DQ-3 | 1つの元コードに、期間が重なる複数の有効な共通機種IDが存在しないこと | HIGH |
| DQ-4 | 機種コード・適用期間が欠損しておらず、期間が逆転していないこと | MEDIUM |
| DQ-5 | 共通機種IDに変換できない実績・計画レコードを把握していること | MEDIUM |